# ResNet50 DPU Benchmark
Measures inference speed and power on the KV260 **FPGA DPU (B512)**.

**Run from Jupyter on the KV260** (`http://192.168.68.60:9090/lab`)

Prerequisites:
- DPU not required to be pre-loaded — `DpuOverlay('dpu.bit')` handles it
- Power sensor: `/sys/class/hwmon/hwmon2/power1_input` (INA260 chip)

In [ ]:
import os, time, numpy as np

POWER_SENSOR = "/sys/class/hwmon/hwmon2/power1_input"
N_WARMUP = 10
N_BENCHMARK = 100

# Smart path: check local models/ first, then pynq default
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("dpu_bench.ipynb"))
XMODEL_LOCAL = os.path.join(NOTEBOOK_DIR, "models", "dpu_resnet50.xmodel")
XMODEL_PYNQ  = "/root/jupyter_notebooks/pynq-dpu/dpu_resnet50.xmodel"
XMODEL = XMODEL_LOCAL if os.path.exists(XMODEL_LOCAL) else XMODEL_PYNQ

def read_power_mw():
    with open(POWER_SENSOR) as f:
        return int(f.read().strip()) / 1000

print(f"Using xmodel: {XMODEL}")
print(f"Current board power: {read_power_mw()/1000:.2f} W (idle)")

## Step 1 — Load the DPU Overlay
`DpuOverlay('dpu.bit')` programs the FPGA with the B512 DPU bitstream.
This takes ~10-20 seconds on first load.

In [ ]:
from pynq_dpu import DpuOverlay

overlay = DpuOverlay("dpu.bit")
print("DPU overlay loaded!")

## Step 2 — Load the ResNet50 Model
The `.xmodel` is a compiled neural network for the DPU — different per model, same `dpu.bit`.

In [ ]:
overlay.load_model(XMODEL)
dpu = overlay.runner

in_t  = dpu.get_input_tensors()
out_t = dpu.get_output_tensors()
print(f"Input shape:  {in_t[0].dims}")
print(f"Output shape: {out_t[0].dims}  (1000 ImageNet classes)")

## Step 3 — Prepare Input Data
Using random data (zeros) — we're measuring hardware speed, not accuracy.

In [ ]:
in_d  = [np.zeros(t.dims, dtype=np.int8) for t in in_t]
out_d = [np.zeros(t.dims, dtype=np.int8) for t in out_t]
print("Input buffers ready")

## Step 4 — Warmup
Run a few frames first to warm up the DPU pipeline.

In [ ]:
print(f"Warming up ({N_WARMUP} frames)...")
for _ in range(N_WARMUP):
    job = dpu.execute_async(in_d, out_d)
    dpu.wait(job)
print("Warmup done!")

## Step 5 — Benchmark
Run 100 frames, measure time and power simultaneously.

In [ ]:
print(f"Benchmarking ({N_BENCHMARK} frames)...")
power_samples = []
start = time.time()
for i in range(N_BENCHMARK):
    job = dpu.execute_async(in_d, out_d)
    dpu.wait(job)
    if i % 10 == 0:
        power_samples.append(read_power_mw())
elapsed = time.time() - start
print("Done!")

## Step 6 — Results

In [ ]:
avg_power_w = sum(power_samples) / len(power_samples) / 1000
fps = N_BENCHMARK / elapsed
latency_ms = elapsed / N_BENCHMARK * 1000
fps_per_watt = fps / avg_power_w

print("=" * 40)
print("RESNET50 DPU BENCHMARK RESULTS")
print("=" * 40)
print(f"Platform:    KV260 DPU (B512)")
print(f"FPS:         {fps:.1f}")
print(f"Latency:     {latency_ms:.1f} ms/frame")
print(f"Power:       {avg_power_w:.2f} W")
print(f"FPS/Watt:    {fps_per_watt:.2f}")
print("=" * 40)

In [ ]:
# Cleanup
del overlay
print("Overlay released")

In [ ]:
import cv2, json, numpy as np
from pynq_dpu import DpuOverlay

# Load labels
LABELS_PATH = "/home/ubuntu/kv260-ubuntu-test/yolox-test/img"
# Try multiple label locations
import glob
label_files = glob.glob("/home/ubuntu/**/labels.json", recursive=True)
if not label_files:
    # Download ImageNet labels if not present
    import urllib.request
    url = "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json"
    urllib.request.urlretrieve(url, "/home/ubuntu/imagenet_labels.json")
    label_files = ["/home/ubuntu/imagenet_labels.json"]
with open(label_files[0]) as f:
    labels = json.load(f)
print(f"Loaded {len(labels)} labels from {label_files[0]}")

# Load and preprocess fish image
# shared/test_image_224.jpg is pre-resized to 224x224
IMAGE_PATHS = [
    "/root/jupyter_notebooks/dpu_benchmark/test_image_224.jpg",
    "/home/ubuntu/test_image_224.jpg",
]
img_path = next((p for p in IMAGE_PATHS if os.path.exists(p)), None)
print(f"Using image: {img_path}")

img = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Normalize: ResNet50 expects mean-subtracted float
mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
img_norm = (img_rgb.astype(np.float32) / 255.0 - mean) / std
# DPU expects (1, 224, 224, 3) int8 — quantize
img_int8 = (img_norm * 128).clip(-128, 127).astype(np.int8)

# Run inference
overlay2 = DpuOverlay("dpu.bit")
overlay2.load_model(XMODEL)
dpu2 = overlay2.runner
in_t2  = dpu2.get_input_tensors()
out_t2 = dpu2.get_output_tensors()
in_d2  = [np.zeros(t.dims, dtype=np.int8) for t in in_t2]
out_d2 = [np.zeros(t.dims, dtype=np.int8) for t in out_t2]

in_d2[0][0] = img_int8
job = dpu2.execute_async(in_d2, out_d2)
dpu2.wait(job)

# Get top-5 predictions
scores = out_d2[0][0].astype(np.float32)
top5_idx = np.argsort(scores)[::-1][:5]

print(f"\nImage: fish.jpg (224x224)")
print("ResNet50 DPU predicts:")
for rank, idx in enumerate(top5_idx):
    label = labels[idx] if idx < len(labels) else f"class_{idx}"
    print(f"  {rank+1}. {label} (score: {scores[idx]:.1f})")

del overlay2

## Bonus — What Does ResNet50 See? (Real Image Test)
Run the fish image through ResNet50 to verify it's doing real inference, not just benchmarking.